## rag 전체 만들어보기

In [ ]:
#!uv run python -c "import sys; print(sys.executable)"

In [3]:
# 환경변수 불러오기

from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
#!uv add langchain-community
#!uv pip install pypdf

In [ ]:
#PDF 파일 읽기
from langchain_community.document_loaders import PyPDFLoader

FILE_PATH = "가상Tech_업무가이드.pdf"

loader = PyPDFLoader(FILE_PATH)

docs = loader.load()

print(len(docs))


9


In [ ]:
#!uv add langchain-text-splitters

In [10]:
# PDF파일의 텍스트 분할
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 30,
)

split_docs = loader.load_and_split(text_splitter=text_splitter)

print(len(split_docs))

31


In [ ]:
#!uv add "langchain-openai"

In [11]:
# PDF chunk의 임베딩 설정
from langchain_openai.embeddings import OpenAIEmbeddings

emeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


In [ ]:
#!uv add langchain-chroma

In [ ]:
from langchain_chroma import Chroma
#db를 생성
db = Chroma.from_documents(
    documents=split_docs,
    embedding=OpenAIEmbeddings(),
    collection_name="my_db"
)

#디스크에 저장되는 chroma vector
DB_PATH = "./chroma_db"
persist_db = Chroma.from_documents(
    split_docs,
    OpenAIEmbeddings(),
    persist_directory=DB_PATH,
    collection_name="my_db"
)

In [23]:
#디스크에서 문서를 로드
persist_db = Chroma(
    persist_directory=DB_PATH,
    embedding_function=OpenAIEmbeddings(),
    collection_name="my_db"
)

In [26]:
# 리트리버 생성
retriever = persist_db.as_retriever( 
    search_type="similarity", 
    search_kwargs={ 
        "k": 3 
    }, 
)

In [27]:
#Openiai LLM 설정

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    temperature=0,
    model_name="gpt-4o-mini",
)

In [ ]:
# #vector store 생성 2번째 방법..?
# from langchain_core.vectorstores import InMemoryVectorStore

# vector_store = InMemoryVectorStore(emeddings)

# #vector store에 저장
# ids = vector_store.add_documents(documents=split_docs)

In [19]:
# 질문
questions = [
    "연차를 사용하려면 최소 며칠 전에 신청해야 하나요?",
    "서울 출장 시 숙박비는 최대 얼마까지 지원되나요?",
    "출장을 다녀온 후 비용 정산은 언제까지 완료해야 하나요?",
    "회사 노트북을 분실했을 경우 어떤 절차로 신고해야 하나요?",
    "재택근무를 신청하려면 어떤 조건을 충족해야 하나요?",
    "직원이 업무 관련 교육을 받을 경우 회사에서 지원하는 비용 기준은 어떻게 되나요?",
    "출장 중 택시비를 회사 경비로 처리할 수 있는 경우는 언제인가요?",
    "개인 사정으로 출근 시간이 늦어질 경우 어떤 방식으로 보고해야 하나요?",
    "업무 중 시스템 장애가 발생했을 때 가장 먼저 해야 하는 조치는 무엇인가요?",
    "제주도로 2박 3일 출장을 가는 경우 숙박비 지원 한도와 출장비 정산 기한을 각각 알려주세요."
]

In [30]:
for i, question in enumerate(questions, start=1):

    # 1. 유사도 검색
    results = vector_store.similarity_search(
        question,
        k=3
    )

    # 2. 검색된 문서 3개를 하나로 합치기
    context = "\n\n".join(
        doc.page_content
        for doc in results
    )

    # 3. LLM에게 전달할 프롬프트
    prompt = f"""
다음 문서를 참고해서 질문에 답변하세요.

[참고 문서]
{context}

[질문]
{question}

문서에 있는 내용만 사용해서 답변하세요.
문서에서 답을 찾을 수 없다면
"문서에서 확인할 수 없습니다."라고 답변하세요.
"""

    # 4. LLM 호출
    response = llm.invoke(prompt)

    
    print(f"질문 {i}: {question}")
    print("답변:", response.content)

질문 1: 연차를 사용하려면 최소 며칠 전에 신청해야 하나요?
답변: 문서에서 확인할 수 없습니다.
질문 2: 서울 출장 시 숙박비는 최대 얼마까지 지원되나요?
답변: 서울 출장 시 숙박비는 1박 15만원(세금 포함)까지 지원됩니다.
질문 3: 출장을 다녀온 후 비용 정산은 언제까지 완료해야 하나요?
답변: 문서에서 확인할 수 없습니다.
질문 4: 회사 노트북을 분실했을 경우 어떤 절차로 신고해야 하나요?
답변: 문서에서 확인할 수 없습니다.
질문 5: 재택근무를 신청하려면 어떤 조건을 충족해야 하나요?
답변: 재택근무를 신청하려면 주 2회의 기준을 충족해야 하며, 매주 월요일까지 캘린더에 ‘WFH’ 표시를 해야 합니다. 문서에서 확인할 수 없습니다.
질문 6: 직원이 업무 관련 교육을 받을 경우 회사에서 지원하는 비용 기준은 어떻게 되나요?
답변: 직원이 업무 관련 교육을 받을 경우 회사에서 지원하는 비용 기준은 연 120만원입니다. 이 지원은 교육, 도서, 컨퍼런스에 사용될 수 있으며, LearnHub에서 사전 승인을 받아야 합니다.
질문 7: 출장 중 택시비를 회사 경비로 처리할 수 있는 경우는 언제인가요?
답변: 출장 중 택시비를 회사 경비로 처리할 수 있는 경우는 다음과 같습니다: 23:00 이후 퇴근, 대중교통 미운행, 고객 일정 등 업무상 필요가 있을 때 실비 지원합니다.
질문 8: 개인 사정으로 출근 시간이 늦어질 경우 어떤 방식으로 보고해야 하나요?
답변: 문서에서 확인할 수 없습니다.
질문 9: 업무 중 시스템 장애가 발생했을 때 가장 먼저 해야 하는 조치는 무엇인가요?
답변: 업무 중 시스템 장애가 발생했을 때 가장 먼저 해야 하는 조치는 감지자가 #help-urgent에 ‘증상, 시작 시각, 영향, 관측 링크’를 게시하고 PagerDuty를 호출하는 것입니다.
질문 10: 제주도로 2박 3일 출장을 가는 경우 숙박비 지원 한도와 출장비 정산 기한을 각각 알려주세요.
답변: 숙박비 지원 한도는 1박 15만원(세금 포함)입니다. 출장비 정산 